# 02 — Example-Based Colorization Pipeline

**Method:** Welsh, Ashikhmin & Mueller — *Transferring Color to Greyscale Images* (SIGGRAPH 2002)  
**Core idea:** Match each grayscale target pixel to the most similar reference pixel using feature vector `f(p) = (L(p), local_std(L_neighborhood(p)))`, then transfer `a, b` chrominance.

---
**Algorithm steps:**
1. Convert target + reference → CIE Lab
2. Build feature vectors `[L, std(L)]` per pixel for both images
3. Build KD-tree on reference features
4. For each target pixel: find k=5 nearest neighbors → pick best by min |ΔL|
5. Transfer `a, b` from best match → combine with target `L` → convert back to BGR

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import cv2
import matplotlib.pyplot as plt
from sklearn.neighbors import KDTree

from src.example_based.colorizer import ExampleColorizer
from src.example_based.matching import build_feature_vectors, kd_tree_match
from src.example_based.utils import bgr_to_lab, lab_to_bgr, local_std

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
print('Imports OK')

In [ ]:
# ── Helper: generate synthetic colorful test images ──────────────────────────
def _make_synthetic(size=256, seed=42):
    """4-quadrant colorful image with slight noise."""
    rng = np.random.RandomState(seed)
    img = np.zeros((size, size, 3), dtype=np.uint8)
    h, w = size // 2, size // 2
    img[:h, :w]  = [30,  60,  200]   # BGR: red-ish
    img[:h, w:]  = [50,  180,  60]   # BGR: green-ish
    img[h:, :w]  = [200, 120,  40]   # BGR: blue-ish
    img[h:, w:]  = [40,  210, 210]   # BGR: yellow-ish
    noise = rng.randint(0, 25, img.shape, dtype=np.uint8)
    return np.clip(img.astype(np.int32) + noise, 0, 255).astype(np.uint8)

# ── Try real COCO images; fall back to synthetic ──────────────────────────────
BENCHMARK_DIR = '../data/raw/coco2017/benchmark'
VAL_DIR       = '../data/raw/coco2017/val2017'

reference_bgr = target_bgr_gt = gray_image = None

for search_dir in [BENCHMARK_DIR, VAL_DIR]:
    if not os.path.isdir(search_dir):
        continue
    jpgs = sorted(f for f in os.listdir(search_dir) if f.lower().endswith('.jpg'))
    if len(jpgs) >= 2:
        target_bgr_gt = cv2.resize(cv2.imread(os.path.join(search_dir, jpgs[0])), (256, 256))
        reference_bgr = cv2.resize(cv2.imread(os.path.join(search_dir, jpgs[1])), (256, 256))
        gray_1ch      = cv2.cvtColor(target_bgr_gt, cv2.COLOR_BGR2GRAY)
        gray_image    = cv2.cvtColor(gray_1ch, cv2.COLOR_GRAY2BGR)
        print(f'Loaded real images from {search_dir}:  {jpgs[0]}  /  {jpgs[1]}')
        break

if target_bgr_gt is None:
    print('No benchmark images found — using synthetic test images.')
    reference_bgr = _make_synthetic(256, seed=0)
    target_bgr_gt = _make_synthetic(256, seed=7)
    gray_1ch      = cv2.cvtColor(target_bgr_gt, cv2.COLOR_BGR2GRAY)
    gray_image    = cv2.cvtColor(gray_1ch, cv2.COLOR_GRAY2BGR)

# ── Display inputs ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, title in zip(
    axes,
    [gray_image, reference_bgr, target_bgr_gt],
    ['Grayscale Target', 'Color Reference', 'Ground Truth'],
):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=11)
    ax.axis('off')
plt.suptitle('Inputs', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Image size: {gray_image.shape[:2]}')

In [ ]:
# ── Run ExampleColorizer ──────────────────────────────────────────────────────
colorizer = ExampleColorizer()
colorized_bgr, info = colorizer.colorize(gray_image, reference_bgr)

print('info_dict:')
for k, v in info.items():
    print(f'  {k}: {v}')

assert colorized_bgr.shape == gray_image.shape, 'Output shape mismatch'
assert colorized_bgr.dtype == np.uint8,          'Output dtype must be uint8'
assert {'method', 'time_seconds', 'image_size'} <= set(info.keys()), 'Missing info keys'
print('\nAll assertions passed.')

In [ ]:
# ── Qualitative comparison: Grayscale / Colorized / Ground Truth ──────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
panels = [
    (gray_image,    'Grayscale Input'),
    (colorized_bgr, f'Colorized — Welsh 2002\n({info["time_seconds"]:.2f} s)'),
    (target_bgr_gt, 'Ground Truth'),
]
for ax, (img, title) in zip(axes, panels):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=11)
    ax.axis('off')
plt.suptitle('Qualitative Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Space: L vs local_std(L) for target and reference ────────────────
L_t, _    = bgr_to_lab(gray_image)
L_r, ab_r = bgr_to_lab(reference_bgr)

std_t = local_std(L_t, window_size=5)
std_r = local_std(L_r, window_size=5)

# Subsample every 6th pixel to reduce density
step  = 6
Lt_s  = L_t[::step, ::step].ravel()
st_s  = std_t[::step, ::step].ravel()
Lr_s  = L_r[::step, ::step].ravel()
sr_s  = std_r[::step, ::step].ravel()

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(Lr_s, sr_s, s=6, alpha=0.35, color='tab:orange', label=f'Reference  ({len(Lr_s):,} px)')
ax.scatter(Lt_s, st_s, s=6, alpha=0.35, color='tab:blue',   label=f'Target     ({len(Lt_s):,} px)')
ax.set_xlabel('L  (luminance, 0–100)', fontsize=12)
ax.set_ylabel('local std(L)  —  texture', fontsize=12)
ax.set_title('Feature Space: f(p) = (L, local\_std(L, 5×5))', fontsize=12)
ax.legend(markerscale=3, fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Target    L: [{Lt_s.min():.1f}, {Lt_s.max():.1f}]   std: [{st_s.min():.2f}, {st_s.max():.2f}]')
print(f'Reference L: [{Lr_s.min():.1f}, {Lr_s.max():.1f}]   std: [{sr_s.min():.2f}, {sr_s.max():.2f}]')

In [ ]:
# ── KD-tree matching demo: show best match for 3 sample pixels ───────────────
H, W = gray_image.shape[:2]

lab_tgt = np.zeros((H, W, 3))
lab_tgt[:, :, 0] = L_t

lab_ref = np.zeros((*L_r.shape, 3))
lab_ref[:, :, 0] = L_r
lab_ref[:, :, 1:] = ab_r

tgt_feats = build_feature_vectors(lab_tgt, neighborhood_size=5)
ref_feats = build_feature_vectors(lab_ref, neighborhood_size=5)
ab_flat   = ab_r.reshape(-1, 2)

tree = KDTree(ref_feats)

sample_pixels = [
    (H // 4,      W // 4,      'top-left'),
    (H // 2,      W // 2,      'center'),
    (3 * H // 4,  3 * W // 4,  'bottom-right'),
]

header = f'{"Pixel":<14} {"L_target":>10} {"std_target":>11} {"L_best":>9} {"dL":>7} {"a_out":>8} {"b_out":>8}'
print(header)
print('-' * len(header))

for r, c, name in sample_pixels:
    idx  = r * W + c
    feat = tgt_feats[idx : idx + 1]         # (1, 2)
    _, inds = tree.query(feat, k=5)          # (1, 5)
    L_cands  = ref_feats[inds[0], 0]         # (5,)
    best_k   = int(np.argmin(np.abs(L_cands - feat[0, 0])))
    best_idx = inds[0, best_k]
    a_out, b_out = ab_flat[best_idx]
    delta_L = abs(L_cands[best_k] - feat[0, 0])
    print(f'{name:<14} {feat[0,0]:>10.2f} {feat[0,1]:>11.2f} {L_cands[best_k]:>9.2f} {delta_L:>7.3f} {a_out:>8.2f} {b_out:>8.2f}')

## Summary

| Property | Value |
|---|---|
| Paper | Welsh, Ashikhmin & Mueller — SIGGRAPH 2002 |
| Feature vector | `f(p) = (L(p), local_std(L, 5×5))` |
| Matching | KD-tree, k=5 candidates, pick by min \|ΔL\| |
| Color transfer | Copy `a, b` from best-matched reference pixel |
| Config defaults | `neighborhood_size=5`, `k_neighbors=5`, `downsample=0.5` |

**Limitations**
- Pixel-level matching with no spatial coherence → may produce noisy color transitions
- Heavily dependent on reference quality — a semantically similar reference gives better colors
- `downsample=0.5` trades slight quality for 4× speed on KD-tree build/query